In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

In [ ]:
# ── 1. CONFIG ────────────────────────────────────────────────
MODEL_ID    = "meta-llama/Meta-Llama-3.1-8B-Instruct"
# MODEL_ID    = "microsoft/Phi-3-mini-4k-instruct"
OUTPUT_DIR  = "./nepali-legal-llama"
RANK        = 16
ALPHA       = 32
LR          = 2e-4
EPOCHS      = 1          # 1 for first run, increase after confirming pipeline works
MAX_SEQ_LEN = 2048
BATCH_SIZE  = 2
GRAD_ACCUM  = 8          # effective batch size = 2 × 8 = 16

In [ ]:
# ── 2. QUANTIZATION CONFIG ───────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)


In [ ]:
from huggingface_hub import login
login()

In [ ]:
# ── 3. LOAD TOKENIZER ────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
# ── 4. LOAD BASE MODEL ───────────────────────────────────────
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False

In [ ]:
model.print_trainable_parameters()

In [ ]:
# ── 5. APPLY LORA ────────────────────────────────────────────
lora_config = LoraConfig(
    r=RANK,
    lora_alpha=ALPHA,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ── 6. LOAD DATASET ──────────────────────────────────────────
# Expected format: ShareGPT JSON with "messages" field
# Each example: {"messages": [{"role": "system", ...}, {"role": "user", ...}, {"role": "assistant", ...}]}
dataset = load_dataset("json", data_files={
    "train":      "train.jsonl",
    "validation": "val.jsonl",
})

In [ ]:
print(dataset.column_names)

In [ ]:
# ── 7. FORMAT WITH CHAT TEMPLATE ─────────────────────────────

def format_example(example):
    messages = [
        {"role": "user", "content": example["question"]},
        {"role": "assistant", "content": example["answer"]},
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    }

dataset = dataset.map(format_example)

In [ ]:
# ── 8. TRAINING CONFIG ───────────────────────────────────────
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="wandb",
    dataset_text_field="text",
)

In [ ]:
# ── 9. TRAIN ─────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
)

trainer.train()


In [ ]:
# ── 10. SAVE ADAPTER ─────────────────────────────────────────
trainer.save_model(f"{OUTPUT_DIR}/final-adapter")

In [ ]:
# ── 11. MERGE AND PUSH ───────────────────────────────────────
merged_model = model.merge_and_unload()

In [ ]:
!zip -r final-adapter.zip {OUTPUT_DIR}/final-adapter
from google.colab import files
files.download("final-adapter.zip")